# 07 · Temporal model and hybrid

**Part A (Step 4): causal history sequences.** Part B (training the GRU model and the Wide & Deep hybrid) follows once the Step 3 training loop exists.

The brief allows a temporal model *only if flight history is correctly ordered*. Here the order is the Phase 1 decision clock: for target date D, a window holds the last 28 **availability days** of outcomes for an origin, a carrier and a route. The newest slot holds outcomes first usable for D, known between 20:00 on D−2 and 20:00 on D−1. Nothing from day D, and nothing from D−1 after 20:00, can enter D's window.

In [1]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import numpy as np, pandas as pd
from aerocast import config as p1, features, leakage, pipeline as p1_pipeline
from aerocast_dl import config, sequences

P = p1_pipeline.prepare()
seqs = sequences.default_sequences(P)
{n: s.windows().shape for n, s in seqs.items()}

{'origin': (365, 3, 28, 5),
 'carrier': (365, 16, 28, 5),
 'route': (365, 224, 28, 5)}

## 1. One window, slot by slot

JFK on 2013-07-11, the newest 7 slots. Each slot is labelled with its **availability date**: the first target date whose decision clock follows the moment the outcome was known. The Phase 1 7-day origin history for any JFK flight that day is exactly the sum of these 7 slots.

In [2]:
D, origin = pd.Timestamp('2013-07-11'), 'JFK'
s = seqs['origin']
w = s.windows()[s.dates.get_loc(D), s.entities.get_loc(origin)]
slots = pd.DataFrame(w[-7:], columns=sequences.CHANNELS,
                     index=pd.date_range(end=D, periods=7, name='usable from'))
ref = features.rolling_past(P.raw, ('origin',), 7, 'is_delayed')
row = P.raw.index[(P.raw['date'] == D) & (P.raw['origin'] == origin)][0]
assert slots['n_arr'].sum() == ref.loc[row, 'n'] and slots['n_late'].sum() == ref.loc[row, 's']
print('Phase 1 7-day JFK history for', D.date(), '->', int(ref.loc[row, 'n']), 'arrivals,', int(ref.loc[row, 's']), 'late')
slots.astype(int)

Phase 1 7-day JFK history for 2013-07-11 -> 2144 arrivals, 722 late


,n_arr,n_late,delay_sum,n_dep,n_cancel
usable from,,,,,
2013-07-05,315,101,4155,309,1
2013-07-06,288,44,-1633,300,2
2013-07-07,307,37,-1124,310,6
2013-07-08,293,66,836,307,6
2013-07-09,325,181,15252,330,10
2013-07-10,313,138,9317,336,5
2013-07-11,303,155,10928,314,21


## 2. Decision clock, inline

Scramble the outcomes of 10 July flights that reached the gate **after** 20:00 on 10 July: the window for 11 July must not move, the one for 12 July must. Scrambling the ones that arrived before 20:00 must move 11 July's window (positive control).

In [3]:
clock = D - pd.Timedelta(days=1) + pd.Timedelta(hours=p1.DECISION_HOUR)
prev = P.raw['date'] == D - pd.Timedelta(days=1)
late, early = prev & (P.raw['arr_known_dt'] > clock), prev & (P.raw['arr_known_dt'] <= clock)

def moved(rows, day):
    s2 = sequences.build_sequences(leakage._scramble_outcomes(P.raw, rows, seed=0))
    k = seqs['origin'].dates.get_loc(day)
    return any(not np.array_equal(seqs[n].windows()[k], s2[n].windows()[k]) for n in seqs)

pd.DataFrame({'flights scrambled': [int(late.sum()), int(early.sum())],
              'window of 11 Jul moved': [moved(late, D), moved(early, D)],
              'window of 12 Jul moved': [moved(late, D + pd.Timedelta(days=1)), None]},
             index=['arrived after 20:00 on 10 Jul', 'arrived before 20:00 on 10 Jul'])

,flights scrambled,window of 11 Jul moved,window of 12 Jul moved
arrived after 20:00 on 10 Jul,292,False,True
arrived before 20:00 on 10 Jul,558,True,None


## 3. Coverage and normalisation

Model inputs per slot: log arrivals, late rate, mean clipped delay, log departures, cancellation rate, and a 0/1 *has arrivals* flag (rates of empty days are 0). The first five are standardised with means and SDs from the windows of the **final training rows only** (`fit_sequence_normalizer`, same guards as the tabular encoder).

In [4]:
summary = json.loads((config.RESULTS_DIR / 'sequence_summary.json').read_text())
for n, s in seqs.items():
    assert list(s.windows().shape) == summary['entity_types'][n]['window_shape']
    assert summary['entity_types'][n]['mismatches_vs_phase1_14d'] == 0
pd.DataFrame(summary['entity_types']).T[['keys', 'entities', 'mean_known_arrivals_per_window', 'modelling_rows_with_empty_window', 'mismatches_vs_phase1_14d']]

,keys,entities,mean_known_arrivals_per_window,modelling_rows_with_empty_window,mismatches_vs_phase1_14d
origin,[origin],3,8234.439759,0.0,0
carrier,[carrier],16,3140.155324,0.000012,0
route,"[origin, dest]",224,281.434677,0.000345,0


## 4. Leakage tests for this step (`tests/test_sequences.py`)

| Test | Asserts |
|---|---|
| `test_sequence_matches_history` | the last 1/7/14/28 slots sum exactly to Phase 1 `rolling_past` for every flight, entity and channel |
| `test_sequence_future_perturbation` | randomising outcomes from a cutoff leaves every window up to the cutoff unchanged; later ones move |
| `test_sequence_late_arrival` | D−1 arrivals after 20:00 reach D+1, not D; earlier ones reach D |
| `test_sequence_cancellation` | a realistic day-D cancellation leaves windows up to D unchanged |
| `test_no_tail_day_counts` | tail-number entities are refused; blanking every tail number changes nothing (G7) |
| `test_sequence_normalizer_train_only` | normaliser statistics ignore every outcome from 1 Sep on |

Mutation check (run once during development): bucketing outcomes by flight date instead of availability date fails 13 of these checks; moving the clock from 20:00 to 23:00 fails the equality and late-arrival tests.